In [72]:
import os
import sys
from pathlib import Path
import time
import json
import warnings
warnings.filterwarnings('ignore')

# Audio processing
import torch
import torchaudio
import numpy as np

# Models
from faster_whisper import WhisperModel

# Vector DB
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# Utilities
from datetime import datetime
from typing import List, Dict, Tuple
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

✅ All imports successful!
PyTorch version: 2.8.0+cu128
CUDA available: False


In [73]:
class Config:
    """Pipeline configuration"""

    # Paths
    AUDIO_FILE = "meeting_sample.mp3"  # Replace with your audio file
    OUTPUT_DIR = Path("./output")

    # Audio settings
    SAMPLE_RATE = 16000

    # Models
    WHISPER_MODEL = "base"  # tiny, base, small, medium, large-v2
    WHISPER_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    WHISPER_COMPUTE_TYPE = "float16" if torch.cuda.is_available() else "int8"
    WHISPER_LANGUAGE = "en" # Set transcription language to English

    # HuggingFace token (required for Pyannote)
    HF_TOKEN = os.getenv("HuggingFacaKey", "YOUR_TOKEN_HERE")

    # Embedding model
    EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"  # Smaller, faster alternative

    # Chunking
    CHUNK_SIZE = 512
    CHUNK_OVERLAP = 64

    # Vector DB
    QDRANT_PATH = "./qdrant_data"
    COLLECTION_NAME = "meeting_test"

In [74]:
config = Config()
config.OUTPUT_DIR.mkdir(exist_ok=True)

print("✅ Configuration set!")
print(f"Audio file: {config.AUDIO_FILE}")
print(f"Device: {config.WHISPER_DEVICE}")
print(f"Whisper language: {config.WHISPER_LANGUAGE}")

✅ Configuration set!
Audio file: meeting_sample.mp3
Device: cpu
Whisper language: en


In [75]:
config.AUDIO_FILE = '/content/WhatsApp Audio 2025-12-06 at 20.23.22_ed3d9725.waptt.opus'

In [76]:
def load_and_preprocess_audio(audio_path: str) -> Tuple[torch.Tensor, int]:
    """Load and preprocess audio file"""
    print(f"📂 Loading audio: {audio_path}")

    # Load audio
    waveform, sample_rate = torchaudio.load(audio_path)

    # Resample if needed
    if sample_rate != config.SAMPLE_RATE:
        print(f"🔄 Resampling from {sample_rate}Hz to {config.SAMPLE_RATE}Hz")
        resampler = torchaudio.transforms.Resample(sample_rate, config.SAMPLE_RATE)
        waveform = resampler(waveform)
        sample_rate = config.SAMPLE_RATE

    # Convert to mono
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    # Normalize
    waveform = waveform / torch.max(torch.abs(waveform))

    duration = waveform.shape[1] / sample_rate
    print(f"✅ Audio loaded: {duration:.2f}s, {sample_rate}Hz")

    return waveform, sample_rate

In [77]:
if Path(config.AUDIO_FILE).exists():
    waveform, sr = load_and_preprocess_audio(config.AUDIO_FILE)
    print(f"Waveform shape: {waveform.shape}")
else:
    print(f"⚠️  Audio file not found: {config.AUDIO_FILE}")
    print("Please update config.AUDIO_FILE with your audio file path")

📂 Loading audio: /content/WhatsApp Audio 2025-12-06 at 20.23.22_ed3d9725.waptt.opus
🔄 Resampling from 48000Hz to 16000Hz
✅ Audio loaded: 5.62s, 16000Hz
Waveform shape: torch.Size([1, 89886])


In [78]:
def load_silero_vad():
    """Load Silero VAD model"""
    print("📥 Loading Silero VAD model...")
    model, utils = torch.hub.load(
        repo_or_dir='snakers4/silero-vad',
        model='silero_vad',
        force_reload=False,
        onnx=False
    )
    get_speech_timestamps = utils[0]
    print("✅ VAD model loaded!")
    return model, get_speech_timestamps

In [79]:
def detect_speech(waveform: torch.Tensor, sample_rate: int):
    """Detect speech segments"""
    print("🎯 Detecting speech segments...")

    vad_model, get_speech_timestamps = load_silero_vad()

    # Ensure 1D tensor
    if waveform.dim() > 1:
        waveform = waveform.squeeze(0)

    speech_timestamps = get_speech_timestamps(
        waveform,
        vad_model,
        sampling_rate=sample_rate,
        threshold=0.5,
        min_speech_duration_ms=250,
        min_silence_duration_ms=100
    )

    # Convert to seconds
    segments = []
    for ts in speech_timestamps:
        segments.append({
            'start': ts['start'] / sample_rate,
            'end': ts['end'] / sample_rate
        })

    print(f"✅ Detected {len(segments)} speech segments")

    # Visualize
    if segments:
        total_speech = sum(s['end'] - s['start'] for s in segments)
        print(f"Total speech time: {total_speech:.2f}s")

    return segments

# Test VAD
if Path(config.AUDIO_FILE).exists():
    vad_segments = detect_speech(waveform, sr)
    print(f"First 3 segments: {vad_segments[:3]}")

🎯 Detecting speech segments...
📥 Loading Silero VAD model...
✅ VAD model loaded!


Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


✅ Detected 3 speech segments
Total speech time: 3.92s
First 3 segments: [{'start': 0.258, 'end': 1.726}, {'start': 2.818, 'end': 4.67}, {'start': 4.802, 'end': 5.406}]


In [80]:
def load_whisper_model():
    """Load Faster-Whisper model"""
    print(f"📥 Loading Whisper model: {config.WHISPER_MODEL}")
    model = WhisperModel(
        config.WHISPER_MODEL,
        device=config.WHISPER_DEVICE,
        compute_type=config.WHISPER_COMPUTE_TYPE
    )
    print("✅ Whisper model loaded!")
    return model

In [81]:
def transcribe_audio(audio_path: str):
    """Transcribe audio file"""
    print("🎤 Transcribing audio...")
    start_time = time.time()

    whisper_model = load_whisper_model()

    segments, info = whisper_model.transcribe(
        audio_path,
        beam_size=5,
        word_timestamps=True,
        vad_filter=True,
        vad_parameters=dict(
            threshold=0.5,
            min_speech_duration_ms=250
        ),
        language=config.WHISPER_LANGUAGE # Specify the language
    )

    # Process segments
    transcription = []
    full_text = []

    for segment in segments:
        seg_dict = {
            'start': segment.start,
            'end': segment.end,
            'text': segment.text.strip(),
            'confidence': segment.avg_logprob,
            'words': []
        }

        # Add word timestamps
        if hasattr(segment, 'words') and segment.words:
            for word in segment.words:
                seg_dict['words'].append({
                    'start': word.start,
                    'end': word.end,
                    'word': word.word,
                    'probability': word.probability
                })

        transcription.append(seg_dict)
        full_text.append(segment.text.strip())

    elapsed = time.time() - start_time

    print(f"✅ Transcription complete!")
    print(f"Language: {info.language} (confidence: {info.language_probability:.2f})")
    print(f"Processing time: {elapsed:.2f}s")
    print(f"Segments: {len(transcription)}")

    return transcription, ' '.join(full_text), info

In [82]:
# Test transcription
if Path(config.AUDIO_FILE).exists():
    transcription, full_text, trans_info = transcribe_audio(config.AUDIO_FILE)

    print("\n📝 Transcription Preview:")
    print("=" * 60)
    for i, seg in enumerate(transcription[:3]):
        print(f"[{seg['start']:.2f}s - {seg['end']:.2f}s]: {seg['text']}")
    print("=" * 60)
    print(f"\nFull text ({len(full_text)} chars): {full_text[:200]}...")


🎤 Transcribing audio...
📥 Loading Whisper model: base
✅ Whisper model loaded!
✅ Transcription complete!
Language: en (confidence: 1.00)
Processing time: 4.71s
Segments: 1

📝 Transcription Preview:
[0.00s - 5.14s]: I wanted to go for a walk, but it started raining heavily on me.

Full text (64 chars): I wanted to go for a walk, but it started raining heavily on me....
